# Tokenization check (Nov 17): summary

Visualizes `summary.csv` from `scripts/tokenization_check.py`: one row per model × bias type (plus `all`),
over StereoSet intrasentence stereotype / anti-stereotype pairs (dev + test).

Columns:
- `pct_both_single_token`: both attribute words are a single token
- `pct_first_diff_mid_word` / `pct_first_diff_other`: first differing token is not the start of the attribute word
- `mean_n_tok_stereo` / `mean_n_tok_anti`: mean attribute-word length in tokens
- `pct_stereo_fewer` / `pct_same` / `pct_stereo_more`: stereotype word has fewer / same / more tokens than the anti-stereotype word

Note: Lapa and Gemma 3 share the Gemma tokenizer (see `meta.json`), so their rows are expected to match.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
df = pd.read_csv(HERE / "summary.csv")
MODELS = list(dict.fromkeys(df["model"]))
BIAS_TYPES = ["all"] + sorted(b for b in df["bias_type"].unique() if b != "all")
df["bias_type"] = pd.Categorical(df["bias_type"], BIAS_TYPES, ordered=True)
df = df.sort_values(["model", "bias_type"]).reset_index(drop=True)
df

## Same tokenizer check
Rows that are identical across models (apart from the model name) indicate a shared tokenizer.

In [ ]:
metrics = [c for c in df.columns if c not in ("model", "bias_type")]
wide = df.pivot(index="bias_type", columns="model", values=metrics)
for i, a in enumerate(MODELS):
    for b in MODELS[i + 1 :]:
        same = np.allclose(wide.xs(a, axis=1, level="model"), wide.xs(b, axis=1, level="model"))
        print(f"{a} vs {b}: {'identical' if same else 'different'}")

## Share of pairs where both attribute words are a single token

In [ ]:
def grouped_bars(ax, value, ylabel, fmt="{:.1f}"):
    x = np.arange(len(BIAS_TYPES))
    w = 0.8 / len(MODELS)
    for i, m in enumerate(MODELS):
        vals = df[df["model"] == m].set_index("bias_type").reindex(BIAS_TYPES)[value]
        bars = ax.bar(x + (i - (len(MODELS) - 1) / 2) * w, vals, w, label=m)
        ax.bar_label(bars, labels=[fmt.format(v) for v in vals], fontsize=7, padding=2)
    ax.set_xticks(x, BIAS_TYPES)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)


fig, ax = plt.subplots(figsize=(9, 4))
grouped_bars(ax, "pct_both_single_token", "% pairs")
ax.set_ylim(0, 100)
ax.set_title("Both attribute words are a single token")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()

## Token-length relation: stereotype vs anti-stereotype word

In [ ]:
parts = [("pct_stereo_fewer", "stereo fewer"), ("pct_same", "same"), ("pct_stereo_more", "stereo more")]
colors = ["tab:blue", "lightgrey", "tab:orange"]

fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4), sharey=True)
for ax, m in zip(np.atleast_1d(axes), MODELS):
    sub = df[df["model"] == m].set_index("bias_type").reindex(BIAS_TYPES)
    bottom = np.zeros(len(sub))
    for (col, label), c in zip(parts, colors):
        ax.bar(BIAS_TYPES, sub[col], bottom=bottom, label=label, color=c)
        for j, v in enumerate(sub[col]):
            if v >= 5:
                ax.text(j, bottom[j] + v / 2, f"{v:.0f}", ha="center", va="center", fontsize=7)
        bottom += sub[col].to_numpy()
    ax.set_title(m)
    ax.tick_params(axis="x", rotation=30)
    ax.spines[["top", "right"]].set_visible(False)
np.atleast_1d(axes)[0].set_ylabel("% pairs")
np.atleast_1d(axes)[-1].legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
fig.suptitle("Token count of stereotype word relative to anti-stereotype word")
fig.tight_layout()

## Mean attribute-word length in tokens

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, col, title in zip(
    axes, ["mean_n_tok_stereo", "mean_n_tok_anti"], ["stereotype word", "anti-stereotype word"]
):
    grouped_bars(ax, col, "mean tokens", fmt="{:.2f}")
    ax.axhline(1, color="grey", lw=0.8, ls="--")
    ax.set_title(title)
hi = df[["mean_n_tok_stereo", "mean_n_tok_anti"]].max().max()
axes[0].set_ylim(0.95, hi + 0.05)
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout()

## Where the first differing token falls
Nonzero values mean the first-difference position is not the start of the attribute word (e.g. shared subtoken prefix).

In [ ]:
cols = ["pct_first_diff_mid_word", "pct_first_diff_other"]
table = df.pivot(index="bias_type", columns="model", values=cols)
table.style.format("{:.1f}").background_gradient(cmap="Reds", vmin=0, vmax=max(1, table.max().max()))

## All metrics at a glance

In [ ]:
heat = df.set_index(["model", "bias_type"])[
    ["pct_both_single_token", "pct_same", "pct_stereo_fewer", "pct_stereo_more",
     "pct_first_diff_mid_word", "pct_first_diff_other"]
]
fig, ax = plt.subplots(figsize=(9, 0.35 * len(heat) + 1))
im = ax.imshow(heat.to_numpy(), cmap="viridis", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(heat.shape[1]), heat.columns, rotation=30, ha="right")
ax.set_yticks(range(len(heat)), [f"{m} / {b}" for m, b in heat.index])
for (i, j), v in np.ndenumerate(heat.to_numpy()):
    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7, color="white" if v < 50 else "black")
fig.colorbar(im, ax=ax, label="%")
fig.tight_layout()